In [ ]:
import os
import shutil

import autokeras as ak
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import mnist


In [ ]:
# Patch for AutoKeras + Keras 3（head.shape 可能含 numpy.int64）
# 僅套用一次；重複執行此格不會堆疊。若先前已重複執行過，請 Restart Kernel 再 Run All。
import autokeras.blocks.heads as _heads

if getattr(_heads, "_CLASSIFICATION_HEAD_BUILD_PATCHED", False):
    print("Patch already applied, skipping")
else:
    _orig_classification_head_build = _heads.ClassificationHead.build

    def _patched_build(self, hp, inputs=None):
        shape = self.__dict__.get("shape")
        if shape is not None:
            self.__dict__["shape"] = tuple(int(x) for x in shape)
        return _orig_classification_head_build(self, hp, inputs)

    _heads.ClassificationHead.build = _patched_build
    _heads._CLASSIFICATION_HEAD_BUILD_PATCHED = True
    print("Patch applied successfully")


In [ ]:
# Keras Tuner 搜尋需要 tensorboard
import tensorboard  # noqa: F401

PROJECT_DIR = "mnist"


def reset_autokeras_project():
    """刪除先前失敗的 trial，避免 evaluate 讀到不完整的 checkpoint。"""
    if os.path.isdir(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
        print(f"已刪除舊專案: {PROJECT_DIR}")

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# ImageClassifier 需要 (樣本數, 高, 寬, 通道)
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)
print("x_train:", x_train.shape, "x_test:", x_test.shape)


In [ ]:
fig = plt.figure()

ax = fig.add_subplot(1 ,2, 1)
plt.imshow(x_train[1234] ,cmap="gray")
ax.set_title(f"Train sample: {y_train[1234]}")

ax = fig.add_subplot(1 ,2, 2)
plt.imshow(x_test[1234] ,cmap="gray")
ax.set_title(f"Test sample: {y_test[1234]}")

plt.show()

In [ ]:
reset_autokeras_project()
clf = ak.ImageClassifier(max_trials=3, project_name=PROJECT_DIR)

In [ ]:
clf.fit(x_train ,y_train ,batch_size=32 ,epochs=1)

In [ ]:
clf.evaluate(x_test ,y_test ,batch_size=32)

In [ ]:
predicted = clf.predict(x_test ,batch_size=32)
predicted

In [ ]:
plt.imshow(x_test[1234] ,cmap="gray")
plt.title(f"Test sample: {y_test[1234]} & Test sample predicted as: {predicted[1234][0]}")
plt.show

In [ ]:
model = clf.export_model()
model.summary()

In [ ]:
from tensorflow.keras.utils import plot_model
plot_model(model ,show_shapes=True)

In [ ]:
# reset_autokeras_project()
# clf = ak.ImageClassifier(max_trials=3, project_name=PROJECT_DIR)

In [ ]:
# clf.fit(x_train ,y_train ,batch_size=32 ,epochs=1)

In [ ]:
# clf.evaluate(x_test ,y_test ,batch_size=32)

In [ ]:
# predicted = clf.predict(x_test ,batch_size=32)
# predicted

In [ ]:
# plt.imshow(x_test[1234] ,cmap="gray")
# plt.title(f"Test sample: {y_test[1234]} & Test sample predicted as: {predicted[1234][0]}")
# plt.show

In [ ]:
# model = clf.export_model()
# model.summary()

In [ ]:
# from tensorflow.keras.utils import plot_model
# plot_model(model ,show_shapes=True)